# Scraping

## Defining scraping functions

### Fonction pour une seule page

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Base URL for pagination
BASE_URL = "https://www.politifact.com/factchecks/list/"

# Function to scrape a single page
def scrape_page(page_number):
    # Parameters for the request
    params = {
        "page": page_number,
        "speaker_type": "SOCIAL"
    }

    # Send HTTP request to the page
    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print(f"Failed to fetch page {page_number}. Status code: {response.status_code}")
        return []

    # Parse the page content with BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all list items with the specified class
    items = soup.find_all('li', class_='o-listicle__item')

    data = []

    for item in items:
        # Extract the headline and post URL
        headline_div = item.find('div', class_='m-statement__quote')
        headline_a = headline_div.find('a') if headline_div else None
        headline = headline_a.text.strip() if headline_a else None
        post_url = "https://www.politifact.com" + headline_a['href'] if headline_a else None

        # Extract the source platform
        meta_div = item.find('div', class_='m-statement__meta')
        source = meta_div.find('a').text.strip() if meta_div else None

        # Extract the fake news rating
        meter_div = item.find('div', class_='m-statement__meter')
        rating_img = meter_div.find('img') if meter_div else None
        rating = rating_img['alt'].strip() if rating_img else None

        # Extract the fact checker and date checked
        footer = item.find('footer', class_='m-statement__footer')
        fact_checker_date = footer.text.strip() if footer else None
        fact_checker = fact_checker_date.split("•")[0].strip() if fact_checker_date else None
        date_checked = fact_checker_date.split("•")[1].strip() if "•" in fact_checker_date else None

        # Extract the post date and type
        description_div = item.find('div', class_='m-statement__desc')
        description_text = description_div.text.strip() if description_div else None
        post_date = description_text.split("stated on")[1].split("in")[0].strip() if "stated on" in description_text else None
        post_type = description_text.split("in")[-1].strip(":").strip() if "in" in description_text else None

        # Add the extracted data to the list
        if headline and source and rating:
            data.append({
                "Headline": headline,
                "Source": source,
                "Rating": rating,
                "Fact Checker": fact_checker,
                "Date Checked": date_checked,
                "Post Date": post_date,
                "Post Type": post_type,
                "Post URL": post_url
            })

    return data

### Fonction pour pages multiples

In [2]:
def scrape_multiple_pages(max_pages=10):
    all_data = []

    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")
        page_data = scrape_page(page)
        if not page_data:
            print(f"No data found on page {page}. Stopping pagination.")
            break
        all_data.extend(page_data)

    return all_data

## Calling scraping functions and making a df

In [3]:
# Scrape data from the first 5 pages (you can adjust this number)
data = scrape_multiple_pages(max_pages=50)

# Convert the data to a Pandas DataFrame
df = pd.DataFrame(data)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...


## Export to csv

In [4]:
df.to_csv("politifact_scraped_data.csv", index=False)

# Nettoyage basique des champs du dataset

## Importation du csv

In [5]:
df = pd.read_csv("politifact_scraped_data.csv")

## Nettoyage avec expressions regulieres et conversions de types

In [6]:
# Remove 'By' from 'Fact Checker' column
df['Fact Checker'] = df['Fact Checker'].str.replace('^By ', '', regex=True)

# Remove posts suffix from source
df['Source'] = df['Source'].str.replace(' posts$', '', regex=True)

# Convert 'Date Checked' and 'Post Date' columns to datetime
df['Date Checked'] = pd.to_datetime(df['Date Checked'], format='%B %d, %Y')
df['Post Date'] = pd.to_datetime(df['Post Date'], format='%B %d, %Y')

# Map ratings to numerical values
rating_map = {
    "true": 0,
    "mostly-true": 0,
    "half-true": 0,
    "barely-true": 1,
    "false": 1,
    "pants-fire": 1
}

# Add a column for numerical ratings
df['Rating'] = df['Rating'].map(rating_map)

## Affichage de rapartition des données

In [7]:
df["Rating"].value_counts()

,count
Rating,
1,1490
0,10


## Export to csv

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Headline      1500 non-null   object        
 1   Source        1500 non-null   object        
 2   Rating        1500 non-null   int64         
 3   Fact Checker  1500 non-null   object        
 4   Date Checked  1500 non-null   datetime64[ns]
 5   Post Date     1500 non-null   datetime64[ns]
 6   Post Type     1500 non-null   object        
 7   Post URL      1500 non-null   object        
dtypes: datetime64[ns](2), int64(1), object(5)
memory usage: 93.9+ KB


In [9]:
df.to_csv("politifact_nettoye.csv", index=False)

# Ajout de 290 vrai news a partir d'une dataset kaggle

## Téléchargement et chargement du dataset dans un df

In [10]:
import kagglehub
import os
import pandas as pd

# Download the dataset from Kaggle
path = kagglehub.dataset_download("sumanthvrao/fakenewsdataset")

# print("Path to dataset files:", path)

# Define the folder containing real news files
real_news_folder = os.path.join(path, "overall", "overall", "real")

# Load all real news articles into a list
real_news = []
for filename in os.listdir(real_news_folder):
    if filename.endswith(".txt"):
        with open(os.path.join(real_news_folder, filename), 'r', encoding='utf-8') as file:
            first_line = file.readline().strip()
            real_news.append({'filename': filename,'Source': 'Kaggle Dataset','Rating':'true', 'Headline': first_line, 'Rating':0})

print(f"Loaded {len(real_news)} real news articles.")
new_real_news_df = pd.DataFrame(real_news)

100%|██████████| 2.60M/2.60M [00:00<00:00, 136MB/s]

Extracting files...


Loaded 290 real news articles.


## Nettoyage du df, suppression des duplications et champs vides

In [11]:
print(f'Df length before cleaning: {len(new_real_news_df)}')

# Recherche et suppression des nuls
# print("\n",new_real_news_df["Headline"].isnull().value_counts(),"\n")
new_real_news_df = new_real_news_df.dropna(subset="Headline")

# Suppression des duplications
new_real_news_df = new_real_news_df.drop_duplicates(subset="Headline").reset_index(drop=True)

# Suppression des Headlines ou il y'a moins de 3 mots
new_real_news_df = new_real_news_df[new_real_news_df["Headline"].str.split().str.len() >= 3]

print(f'Df length after cleaning: {len(new_real_news_df)}')

Df length before cleaning: 290
Df length after cleaning: 284


## Suppression du colonne filename

In [12]:
new_real_news_df = new_real_news_df.drop(axis=1,columns=["filename"])
new_real_news_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 284 entries, 0 to 288
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Source    284 non-null    object
 1   Rating    284 non-null    int64 
 2   Headline  284 non-null    object
dtypes: int64(1), object(2)
memory usage: 8.9+ KB


## Importation du csv

In [13]:
import pandas as pd

data = pd.read_csv("politifact_nettoye.csv")

## Concat avec le df existant

In [14]:
data = pd.concat([data, new_real_news_df], ignore_index=True)

## Visualisation des classes

In [15]:
data["Rating"].value_counts()

,count
Rating,
1,1490
0,294


In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1784 entries, 0 to 1783
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Headline      1784 non-null   object
 1   Source        1784 non-null   object
 2   Rating        1784 non-null   int64 
 3   Fact Checker  1500 non-null   object
 4   Date Checked  1500 non-null   object
 5   Post Date     1500 non-null   object
 6   Post Type     1500 non-null   object
 7   Post URL      1500 non-null   object
dtypes: int64(1), object(7)
memory usage: 111.6+ KB


## Export to csv

In [17]:
data.to_csv("politifact_merged_with_dataset.csv", index=False)

# Augmentation des vrai news dans le dataset

## Par backtranslation

### Installation du module googletrans

In [18]:
!pip install googletrans==4.0.0rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.8 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17397 sha256=c6ee688a376e77426378652882c61c37f4233c32ec12ae42c0cc5103b714b9cc
  Stored in directory: /root/.cache/pip/wheels/c0/59/9f/7372f0cf70160fe61b528532e1a7c8498c4becd6bcffb022de
Successfully built googletrans
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Succ

### Definition de fonction

In [19]:
from googletrans import Translator

# Initialize the Google Translator
translator = Translator()

def backtranslate(text, src_lang='en', target_lang='fr'):
    try:

        # Step 1: Translate to the target language
        print(f"🌐 Translating '{text}' from {src_lang} to {target_lang}...")
        translated = translator.translate(text, src=src_lang, dest=target_lang).text

        # Step 2: Translate back to the source language
        back_translated = translator.translate(translated, src=target_lang, dest=src_lang).text
        # Check for unwanted "Nan" string or invalid result
        if back_translated.strip().lower() == "nan":
            print("❌ Backtranslation returned 'Nan'. Skipping.")
            return None  # Return None to exclude this entry
        print(f"✅ Backtranslated text ({target_lang} -> {src_lang}): '{back_translated}'")
        return back_translated

    except Exception as e:
        print(f"❌ Error during backtranslation for text: '{text}'\nError: {e}")
        return None  # Return original text if there's an error



### Importation du csv

In [20]:
import pandas as pd

df = pd.read_csv("politifact_merged_with_dataset.csv")

### Appel de fonction d'augmentation

In [21]:
# Filter rows where Rating == 0
real_news = df[df['Rating'] == 0]['Headline']

# Apply backtranslation on real news headlines
backtranslated_real_news = real_news.apply(lambda x: backtranslate(x)).dropna()

# Create a new DataFrame with backtranslated headlines
backtranslated_df = pd.DataFrame({
    'Source': 'Kaggle Dataset',
    'Rating':'true',
    'Headline': backtranslated_real_news,
    'Rating': 0  # Assign the same label (0 for real news)
})

🌐 Translating 'FEMA delivered “only four” temporary housing units to Helene victims in North Carolina as of Nov. 20.' from en to fr...
✅ Backtranslated text (fr -> en): 'The FEMA delivered "only four" temporary housing to the victims of Hélène in North Carolina to November 20.'
🌐 Translating 'A Pennsylvania TV station aired the 2024 presidential election results early.' from en to fr...
✅ Backtranslated text (fr -> en): 'A Pennsylvania television channel released the results of the 2024 presidential elections early.'
🌐 Translating 'Project 2025 would “cut” the Federal Emergency Management Agency and “kill federal-backed flood insurance.”' from en to fr...
✅ Backtranslated text (fr -> en): 'The 2025 project "would reduce" the Federal Emergency Management Agency and "would kill the flood insurance supported by the federal government".'
🌐 Translating 'Florida "put Tom Walz instead of Tim Walz" on its ballots.' from en to fr...
✅ Backtranslated text (fr -> en): 'Florida "put Tom Walz inste

### Concatenation avec la dataset initiale

In [22]:
# Merge backtranslated headlines with the original DataFrame
augmented_df = pd.concat([df, backtranslated_df], ignore_index=True)

# Shuffle the dataset to mix augmented and original data
augmented_df = augmented_df.sample(frac=1, random_state=42).reset_index(drop=True)

### Suppression des duplications

In [23]:
# Suppression des duplications
augmented_df = augmented_df.drop_duplicates(subset="Headline").reset_index(drop=True)

### Affichage des classes

In [24]:
augmented_df["Rating"].value_counts()

,count
Rating,
1,1490
0,581


### Export to csv

In [25]:
augmented_df.to_csv("politifact_augmented_backtranslation.csv", index=False)

## Par remplacement de synonyme

### Definition de fonction

In [33]:
import random
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet')

# Define the synonym replacement function
def synonym_replacement(sentence, n=1):
    if type(sentence)==str:
      words = sentence.split()
      for _ in range(n):
          word = random.choice(words)  # Randomly choose a word
          synonyms = wordnet.synsets(word)  # Get synonyms for the word
          if synonyms:
              synonym = synonyms[0].lemmas()[0].name()  # Pick the first synonym
              sentence = sentence.replace(word, synonym, 1)  # Replace the word
    return sentence

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Importation du csv

In [34]:
import pandas as pd

df = pd.read_csv("politifact_augmented_backtranslation.csv")

### Appel de fonction d'augmentation

In [35]:
# Apply synonym replacement only to rows with Rating == 0 in the original DataFrame
df_augmented = df[df['Rating'] == 0].copy()  # Create a new DataFrame for augmented data

# Apply synonym replacement to the headlines
df_augmented['Headline'] = df_augmented['Headline'].apply(lambda x: synonym_replacement(x))

# Combine the original DataFrame with the augmented data (only those with Rating == 0)
merged_df = pd.concat([df, df_augmented], ignore_index=True)

### Suppression des duplications

In [36]:
# Affichage et suppression des duplications
print(f"Taille du dataset augmenté avant suppression des duplications: {len(merged_df)}\n")

print(f"Nombre de Headlines dupliqués aprés augmentation: \n{merged_df.duplicated().value_counts()}\n")
merged_df = merged_df.drop_duplicates(subset="Headline").reset_index(drop=True)

print(f"Taille du dataset augmenté aprés suppression des duplications: {len(merged_df)}")

Taille du dataset augmenté avant suppression des duplications: 2652

Nombre de Headlines dupliqués aprés augmentation: 
False    2306
True      346
Name: count, dtype: int64

Taille du dataset augmenté aprés suppression des duplications: 2306


### Affichage des classes

In [37]:
merged_df["Rating"].value_counts()

,count
Rating,
1,1490
0,816


### Export to csv

In [38]:
merged_df.to_csv("politifact_augmented_synonym.csv", index=False)

# Pretraitement avancé avec NLTK et TextBlob

## Preparation

### Install libraries

In [39]:
!pip install textblob nltk scikit-learn pandas

### Import libraries and download ressources

In [40]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Define preprocessing function using NLTK

In [41]:
# Preprocessing function
def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove punctuation and numbers
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    # Join tokens back into a single string
    return " ".join(tokens)

## Importation du csv

In [42]:
data = pd.read_csv("politifact_augmented_synonym.csv")

## Pretraitement avec fonction basé sur nltk et textblob

In [43]:
# Apply preprocessing to the "Headline" column
data["Headline"] = data["Headline"].apply(preprocess_text)

# Add sentiment analysis features
data["Polarity"] = data["Headline"].apply(lambda x: TextBlob(x).sentiment.polarity)
data["Subjectivity"] = data["Headline"].apply(lambda x: TextBlob(x).sentiment.subjectivity)

## Visualisation avant suppression des duplications

In [44]:
data["Rating"].value_counts()

,count
Rating,
1,1490
0,816


## Suppression des duplications

In [45]:
# Affichage et suppression des duplications
print(f"Taille du dataset avant suppression des duplications: {len(data)}\n")

print(f"Nombre de Headlines dupliqués aprés pretraitement: \n{data.duplicated(subset='Headline').value_counts()}\n")
data = data.drop_duplicates(subset="Headline").reset_index(drop=True)

print(f"Taille du dataset aprés suppression des duplications: {len(data)}")

Taille du dataset avant suppression des duplications: 2306

Nombre de Headlines dupliqués aprés pretraitement: 
False    2160
True      146
Name: count, dtype: int64

Taille du dataset aprés suppression des duplications: 2160


## Visualisation aprés pretraitement et suppression des duplications

In [46]:
data["Rating"].value_counts()

,count
Rating,
1,1488
0,672


## Export to csv

In [47]:
data.to_csv("politifact_clean.csv", index=False)

# Apprentissage du model

## Importation du csv

In [48]:
data = pd.read_csv("politifact_clean.csv")

## Vectorisation avec TF-IDF

In [49]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Vectorize the text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(data["Headline"]).toarray()

## Splitting des données

In [50]:
# Define the target variable using Rating
y = data["Rating"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Apprentissage du modele

In [51]:
# Train a Random Forest Classifier
model = RandomForestClassifier(n_estimators=250,random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=250, random_state=42)

## Evaluation du modele

In [52]:
# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=[
    "True", "False"
]))

Accuracy: 0.9189814814814815

Classification Report:
               precision    recall  f1-score   support

        True       0.98      0.74      0.84       127
       False       0.90      0.99      0.95       305

    accuracy                           0.92       432
   macro avg       0.94      0.87      0.89       432
weighted avg       0.92      0.92      0.92       432



## Prediction manuelle du modèle

In [56]:
headline = "When Donald Trump lost the Iowa caucus to Ted Cruz in 2016, 'he immediately cried that Cruz cheated.'"
# headline = "Super Mario Run makes leap to Android"
text = preprocess_text(headline)
print(text)
sentence = vectorizer.transform([text]).toarray()

prediction = model.predict(sentence)
print("Prediction:", prediction)

# Convert the prediction (0 or 1) to "True" or "False"
if prediction == 0:
    print(f"The headline is classified as: True")
else:
    print(f"The headline is classified as: Fake")

donald trump lost iowa caucus ted cruz immediately cried cruz cheated
Prediction: [1]
The headline is classified as: Fake


## Enregistrement du modele

In [54]:
import joblib

# Save the model and the vectorizer
joblib.dump(model, 'random_forest_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']

# Code Flask
# Il faut l'executer en local. In ne fonctionne pas sur Colab

## app.py

In [ ]:
from flask import Flask, render_template, request
import joblib
import numpy as np

# Initialize the Flask app
app = Flask(__name__)

# Load the model and the vectorizer
model = joblib.load('random_forest_model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

@app.route("/", methods=["GET", "POST"])
def index():
    prediction = None
    if request.method == "POST":
        # Get the text input from the user
        headline = request.form["headline"]

        # Transform the text input using the vectorizer
        transformed_text = vectorizer.transform([headline]).toarray()

        # Make a prediction using the model
        prediction = model.predict(transformed_text)[0]

        # Convert the numeric prediction to a more understandable label
        if prediction == 0:
            prediction = "Real"
        else:
            prediction = "Fake"

    return render_template("index.html", prediction=prediction)

if __name__ == "__main__":
    app.run(debug=False)


## templates

### index.html

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Fake Social Media Posts Detector</title>
    <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-100 text-gray-800">

    <!-- Page Container -->
    <div class="min-h-screen flex items-center justify-center">
        <div class="bg-white shadow-lg rounded-lg p-8 w-full max-w-md">
            <!-- Title -->
            <h1 class="text-3xl font-bold text-center mb-6 text-blue-600">Fake Social Media Posts</h1>

            <!-- Form -->
            <form method="POST" class="space-y-4">
                <div>
                    <label for="headline" class="block text-gray-700 font-semibold mb-2">Enter a headline:</label>
                    <input
                        type="text"
                        id="headline"
                        name="headline"
                        required
                        class="w-full px-4 py-2 border border-gray-300 rounded-md focus:ring-2 focus:ring-blue-500 focus:outline-none"
                        placeholder="e.g., Breaking news headline...">
                </div>
                <div>
                    <button
                        type="submit"
                        class="w-full bg-blue-600 text-white font-semibold py-2 rounded-md hover:bg-blue-700 transition duration-300">
                        Submit
                    </button>
                </div>
            </form>

            <!-- Prediction Result -->
            {% if prediction %}
            <div class="mt-6">
                <h2 class="text-xl font-semibold text-center">
                    Prediction:
                    <span class="{% if prediction == 'Real' %}text-green-600{% else %}text-red-600{% endif %}">
                        {{ prediction | capitalize }}
                    </span>
                </h2>
            </div>
            {% endif %}
        </div>
    </div>
</body>
</html>